In [1]:
import numpy as np
from sympy import *

In [2]:
# Defini variaveis
init_printing()
n = 2
varls = var('x1:3')

In [3]:
# Defini a função
f = x1**4+2*x2*x1**2+x2**2-4*x1
fl = Lambda(varls, f)
fl

In [4]:
# Função gradiente e hessiano
def gradient_op(f, variables):
    return Matrix([Lambda(variables, f.diff(v)) for v in variables])

def hesiano_op(f, variables):
    return Matrix([[Lambda(variables, f.diff(v,u)) for u in variables] for v in variables])

In [5]:
grad = gradient_op(f, varls)
grad

⎡               3              ⎤
⎢(x₁, x₂) ↦ 4⋅x₁  + 4⋅x₁⋅x₂ - 4⎥
⎢                              ⎥
⎢                  2           ⎥
⎣   (x₁, x₂) ↦ 2⋅x₁  + 2⋅x₂    ⎦

In [6]:
hes = hesiano_op(f, varls)
hes

⎡             ⎛    2     ⎞                 ⎤
⎢(x₁, x₂) ↦ 4⋅⎝3⋅x₁  + x₂⎠  (x₁, x₂) ↦ 4⋅x₁⎥
⎢                                          ⎥
⎣     (x₁, x₂) ↦ 4⋅x₁        (x₁, x₂) ↦ 2  ⎦

In [7]:
# Funções para calcular gradiente e hesiano
def gradient_res(gradient, vector):
    results = []
    for dx in gradient:
        results.append(dx(*vector))
    return results

def hesiano_res(hesiano, vector):
    matrix = []
    lin = []
    arr = np.array(hesiano)
    for i in arr:
        for j in i:
            lin.append(j(*vector))
        matrix.append(lin)
        lin = []
    return Matrix(matrix)

def modulo(x): 
    return float(sqrt(sum(i**2 for i in x)))

In [8]:
# Defini x0 e calcula grad e hesiano no ponto
x = [1,0]

In [9]:
gradl = gradient_res(grad, x)
gradl

In [10]:
hesl = hesiano_res(hes, x)
hesl

⎡12  4⎤
⎢     ⎥
⎣4   2⎦

In [11]:
# Testando o calculo do modulo
modulo(gradl)

In [12]:
# Defini as constantes do método
sig = 0.1
alp = 0.1
gama = 0.5
the = 0.1
beta = 0.1
M = 1000
epi = 0.1

In [13]:
# Criando as iteracoes
k = 0
while modulo(gradient_res(grad, x)) > epi and k < M:
    mi = 0
    gradl = np.array(gradient_res(grad, x), dtype=np.float)
    hesl =  np.array(hesiano_res(hes, x), dtype=np.float)
    
    #montando sistema para achar dk
    A = [] 
    while True:
        A = hesl + mi*np.identity(n, int)
        try:
            d = np.linalg.solve(A, -gradl)
        
            if sum(d*gradl) > -the*modulo(gradl)*modulo(d):
                mi = max(2*mi, beta)
            else:
                break
        except:
            mi = max(2*mi, beta)
            
    #fazendo as verificações de armijo
    if modulo(d) < sig*modulo(gradl):
        d = the*(modulo(gradl)/modulo(d))*d
        
    #calculando xk
    t = 1
    while fl(*(x+d*t)) > fl(*x) + alp*t*sum(d*gradl):
        t = gama*t
    
    x = x+d*t
    k = k + 1

In [14]:
x, fl(*x)

(array([   43.81800555, -1919.99526548]), -175.271522904746)